# Sentiment Analysis with HuggingFace — IMDb Movie Reviews

**Course**: AI for Climate & NatureTech  
**Instructor**: Dr. Omer Tzuk

In this notebook we'll learn how to use **HuggingFace Transformers** for sentiment analysis — classifying text as *positive* or *negative*.

**Why does this matter for conservation?**  
The same NLP pipeline we build here for movie reviews can be applied to:
- Analyzing public sentiment toward conservation policies
- Mining social media for attitudes toward wildlife and protected areas
- Processing field reports and stakeholder feedback at scale

We use the **IMDb Reviews Dataset** (50,000 labeled movie reviews) as our training ground.

**What you'll learn:**
1. Loading & exploring a labeled text dataset
2. Using a **pre-trained** model (zero coding effort!)
3. **Fine-tuning** a model on our own data
4. Evaluating model performance

# Setup

## Install dependencies

In [ ]:
# Install required libraries (run once)
!pip install -q transformers datasets accelerate scikit-learn

In [ ]:
# Optional: Authenticate with HuggingFace to silence warnings
# 1. Go to https://huggingface.co/settings/tokens
# 2. Create a free account if you don't have one
# 3. Generate a token (read access is enough)
# 4. In Colab: click the 🔑 key icon in the left sidebar
#    → Add a new secret named HF_TOKEN with your token value

import os
from google.colab import userdata

try:
    os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
    print("✅ HuggingFace token loaded")
except Exception:
    print("⚠️ No HF_TOKEN found — continuing without authentication (this is fine for public models)")

## Imports

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

## Load IMDB dataset

In [ ]:
df = pd.read_csv("IMDB-Dataset.csv", on_bad_lines='skip', engine='python')
print(f"Dataset shape: {df.shape}")
df.head()

# Exploring the data

In [ ]:
print(df['sentiment'].value_counts())
print()

# Sample reviews
for sentiment in ['positive', 'negative']:
    sample = df[df['sentiment'] == sentiment].iloc[0]
    print(f"--- {sentiment.upper()} review (first 300 chars) ---")
    print(sample['review'][:300])
    print()

## Review length distribution

In [ ]:
df['review_length'] = df['review'].str.len()

fig, ax = plt.subplots(1, 1, figsize=(8, 4))
df.groupby('sentiment')['review_length'].plot.hist(
    bins=50, alpha=0.6, ax=ax, legend=True
)
ax.set_xlabel("Review length (characters)")
ax.set_title("Distribution of Review Lengths by Sentiment")
plt.tight_layout()
plt.show()

# Part A: Using a Pre-Trained Model (No Training Needed!)

HuggingFace's `pipeline` lets us run sentiment analysis in **3 lines of code**.  
The model was already trained on large datasets — we just apply it to our data.

This is analogous to **transfer learning** in ecology:  
a species classifier trained on ImageNet can recognize animals it was never explicitly taught.

In [ ]:
from transformers import pipeline

# Load a pre-trained sentiment analysis model
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
    device=0  # Use GPU if available; change to -1 for CPU
)

# Test on a few examples
test_texts = [
    "This movie was absolutely wonderful and moving!",
    "Terrible film. Complete waste of time.",
    "The movie was okay, nothing special."
]

results = sentiment_pipeline(test_texts)
for text, result in zip(test_texts, results):
    print(f"Text: {text}")
    print(f"  → {result['label']} (confidence: {result['score']:.3f})\n")

In [ ]:
# Evaluate on a random subset (full dataset would take a while)
sample_df = df.sample(n=10, random_state=42).reset_index(drop=True)

# The pipeline has a max token limit — truncate long reviews
predictions = sentiment_pipeline(
    sample_df['review'].tolist(),
    truncation=True,
    max_length=512,
    batch_size=32
)

# Map predictions to match our label format
pred_labels = [p['label'].lower() for p in predictions]
true_labels = sample_df['sentiment'].tolist()

print(classification_report(true_labels, pred_labels))

# Part B: ClimateBERT — Detecting Environmental Claims (Greenwashing Detector)

## Introduction to the model

### Why do we need a climate-specific language model?

The sentiment analysis model we used in Part A was trained on general English text (movie reviews). But **climate-related language is a specialized domain** with its own vocabulary: terms like "Scope 1 emissions", "TCFD disclosures", "net-zero pathways", and "stranded assets" appear frequently in corporate sustainability reports and climate science, but rarely in general text. When a general-purpose model encounters such text, it often misunderstands the meaning — just as a general field guide would struggle in a highly specialized ecosystem.

### What is ClimateBERT?

**ClimateBERT** (Webersinke et al., 2022) addresses this gap through **domain-adaptive pre-training** — taking an existing language model and continuing to train it on domain-specific text. The process follows three stages:

1. **Start with a general model**: DistilRoBERTa — a compact, efficient version of RoBERTa
2. **Further pre-train on climate text**: Over **2 million paragraphs** of climate-related content collected from news articles, research paper abstracts, and corporate climate reports (sustainability reports, CDP disclosures, TCFD reports)
3. **Fine-tune for specific tasks**: The resulting base model is then adapted to individual downstream tasks

This approach yielded a **48% improvement** on the model's ability to understand climate language, translating into **3.5%–35.7% lower error rates** across downstream tasks compared to general-purpose models.

This is the same principle behind **transfer learning** in ecology: a model pretrained on general images (ImageNet) can be fine-tuned on a small set of camera trap photos to identify local species — because it already understands visual features like edges, textures, and shapes. Here, ClimateBERT already "speaks climate", so even a small fine-tuning dataset is enough to achieve strong performance.

### The ClimateBERT model family

The research team at ETH Zurich and the University of Zurich released a family of fine-tuned models, each targeting a different real-world need:

| Model | What it does |
|-------|-------------|
| **Environmental Claims Detection** | Is this sentence making a real environmental claim? |
| **Climate Sentiment** | Does a paragraph frame climate as a risk, opportunity, or neutral? |
| **Climate Detection** | Is this paragraph about climate change at all? |
| **TCFD Classification** | Which TCFD disclosure category does this paragraph belong to? |
| **Specificity Detection** | Is a climate statement specific and verifiable, or vague? |
| **Net-Zero / Reduction** | Does this sentence contain a net-zero or emission reduction target? |

### What we'll do: Greenwashing detection

We'll use the **environmental claims detector** — a model fine-tuned on **2,647 expert-annotated sentences** from corporate sustainability reports, annual reports, and earnings calls (Stammbach et al., 2023). It classifies each sentence as either an **environmental claim** (suggesting a product, service, or company is environmentally friendly) or **not a claim** (technical descriptions, generic business language, or vague sustainability wording).

This is directly applicable to **greenwashing detection**: distinguishing genuine sustainability commitments from vague marketing language. When the researchers applied this model to 12 million sentences from earnings calls of 3,361 companies (2012–2019), they found that environmental claims **doubled after the Paris Agreement (2015)** — but more claims don't necessarily mean more action.

**For sustainability professionals**, this means: automatically scanning corporate reports at scale, monitoring ESG disclosure trends over time, and flagging gaps between stated commitments and vague language.

> 📖 **ClimateBERT base model**: Webersinke et al. (2022), *"ClimateBERT: A Pretrained Language Model for Climate-Related Text"* ([arXiv:2110.12010](https://arxiv.org/abs/2110.12010))
>
> 📖 **Environmental claims dataset**: Stammbach et al. (2023), *"Environmental Claim Detection"* ([arXiv:2209.00507](https://arxiv.org/abs/2209.00507))
>
> 🏠 **All models**: [huggingface.co/climatebert](https://huggingface.co/climatebert) | **This model**: [climatebert/environmental-claims](https://huggingface.co/climatebert/environmental-claims)

**Important lesson ahead:** Different HuggingFace models use different label conventions. The sentiment model in Part A used `POSITIVE`/`NEGATIVE`. ClimateBERT uses `yes`/`no`. Always check `model.config.id2label` first!


## Loading the *model*

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline
import pandas as pd

tokenizer_name = "climatebert/distilroberta-base-climate-f"
model_name = "climatebert/environmental-claims"

model = AutoModelForSequenceClassification.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(tokenizer_name, max_len=512)

# IMPORTANT: Always check label mapping!
print("Model label mapping:", model.config.id2label)
# Expected: {0: 'no', 1: 'yes'}
# This means the pipeline returns 'yes' or 'no' — NOT 'LABEL_0'/'LABEL_1'

claims_pipeline = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device=0
)

## Test on examples

In [ ]:
# Examples modeled on the paper's annotation guidelines (Stammbach et al. 2023)
# Training data: sentences from sustainability reports, annual reports, earnings calls
# Environmental claims: suggest a product/service/company is environmentally friendly

test_statements = [
    # --- ENVIRONMENTAL CLAIMS (from paper Tables 4 & 6, or closely modeled) ---

    # Clear claims: specific environmental benefit stated
    "Hydro has also started working on several initiatives to reduce direct CO2 emission in primary aluminium production.",
    "We plan to continue our low risk growth strategy by building our core business with rate base infrastructure, while maintaining the commitment to renewable energy initiatives and to reducing emissions.",
    "Farmers who operate under this scheme are required to dedicate 10% of their land to wildlife preservation.",
    "The company has committed to reduce its carbon footprint by 50% by 2030, and also achieve net zero emissions by 2050.",
    "Our products are manufactured using 100% recycled aluminium, reducing energy consumption by 95% compared to primary production.",
    "A total population of 6148 is getting the benefit of safe potable drinking water due to this initiative.",

    # --- NOT ENVIRONMENTAL CLAIMS (from paper Tables 5 & 6, or closely modeled) ---

    # Technical descriptions (no claim of positive impact)
    "Emissions are modelled based on sector averages including linear regression and country carbon emissions intensities for GDP.",
    "Our key sources of emissions are the running of our operations, purchased goods and services, and land leased to sheep and beef farming.",

    # Generic business language (no environmental content)
    "Teams are thus focused on a shared objective in terms of growth and value creation.",
    "So there is an annual cycle that to some degree dictates the pace of these enrollment campaigns.",

    # Vague sustainability language (borderline — the paper's hardest category)
    "This will further accelerate the company's positive impact by creating and delivering solutions to tackle some of the biggest challenges the world is facing.",
    "We are committed to creating a sustainable future for all our stakeholders and communities worldwide.",
]

expected = [
    True, True, True, True, True, True,  # claims
    False, False,                          # technical descriptions
    False, False,                          # generic business
    False, False,                          # vague sustainability
]

results = claims_pipeline(test_statements, truncation=True, padding=True)

print("=" * 75)
print("ENVIRONMENTAL CLAIMS DETECTOR — Results")
print("=" * 75)

correct = 0
for text, result, expect in zip(test_statements, results, expected):
    is_claim = result['label'] == 'yes'  # ← ClimateBERT uses 'yes'/'no', not 'LABEL_1'/'LABEL_0'
    confidence = result['score']
    icon = "✅" if is_claim else "❌"
    match = "✓" if is_claim == expect else "✗ MISMATCH"

    print(f"\n\"{text[:85]}{'...' if len(text) > 85 else ''}\"")
    print(f"  → {icon} {'ENVIRONMENTAL CLAIM' if is_claim else 'NOT a claim'} "
          f"(confidence: {confidence:.3f})  [{match}]")

    if is_claim == expect:
        correct += 1

print(f"\n{'=' * 75}")
print(f"Accuracy on curated examples: {correct}/{len(expected)} ({100*correct/len(expected):.0f}%)")

### What did we just see?

The model distinguishes between:
- **Environmental claims** — statements with specific, verifiable environmental content (emission reductions, renewable targets, recycling, wildlife preservation)
- **Non-claims** — which include:
  - *Technical descriptions* — mentioning emissions or environmental topics factually, without claiming positive impact
  - *Generic business language* — no environmental content at all
  - *Vague sustainability language* — aspirational wording without specific commitments

This maps directly to the **greenwashing problem**: companies often use vague environmental language to appear green without making concrete commitments. The ClimateBERT research team found that environmental claims in corporate earnings calls **steadily increased after the Paris Agreement (2015)** — but not all claims are equally substantive.

### A practical lesson: label conventions

Notice that in Part A, the sentiment model returned `POSITIVE`/`NEGATIVE`, but ClimateBERT returns `yes`/`no`. This is because each model author defines their own label mapping in `model.config.id2label`. **Always check this mapping** when using a new model — it's a common source of bugs!

## Apply to a batch of corporate statements

In [ ]:
corporate_statements = pd.DataFrame({
    'statement': [
        # Modeled on paper's style: formal corporate disclosure sentences, 10-40 words
        "The company achieved carbon neutrality in its direct operations through a combination of energy efficiency measures and verified carbon offset projects.",
        "We believe in the power of innovation to address the challenges facing our industry and our customers globally.",
        "Total water withdrawal across all manufacturing sites decreased by 12% compared to the previous reporting year.",
        "Our corporate strategy remains aligned with the long-term goals set forth by the board of directors.",
        "We installed 45 MW of on-site solar capacity across our manufacturing facilities, displacing approximately 60,000 tonnes of CO2 annually.",
        "The company is dedicated to responsible growth and creating shared value for all our stakeholders.",
        "Methane emissions from our natural gas operations were reduced by 28% since 2020 through our leak detection and repair programme.",
        "Revenue for the quarter increased by 8% year over year, driven by strong demand across all business segments.",
        "We offset 150,000 tonnes of CO2 through verified reforestation projects certified under the Gold Standard.",
        "Protecting biodiversity remains an important consideration in our ongoing operations and supply chain management.",
    ],
    'source': [
        'Sustainability Report', 'CEO Letter', 'Sustainability Report', 'Strategy Document',
        'Technical Report', 'Marketing Material', 'ESG Disclosure', 'Earnings Call',
        'Carbon Report', 'Annual Report'
    ]
})

results = claims_pipeline(
    corporate_statements['statement'].tolist(),
    truncation=True,
    padding=True
)

corporate_statements['is_claim'] = [r['label'] == 'yes' for r in results]
corporate_statements['confidence'] = [r['score'] for r in results]

print(f"Out of {len(corporate_statements)} statements:")
print(f"  ✅ Environmental claims: {corporate_statements['is_claim'].sum()}")
print(f"  ❌ Non-claims:           {(~corporate_statements['is_claim']).sum()}")
print()

for _, row in corporate_statements.iterrows():
    icon = "✅" if row['is_claim'] else "❌"
    print(f"{icon} [{row['source']}] \"{row['statement'][:75]}...\"")
    print(f"   Confidence: {row['confidence']:.3f}")
    print()

# Part C: Bird Species Classification from Images

In Parts A and B we classified **text**. Now we'll classify **images** — using the exact same HuggingFace `pipeline` API.

We'll use a model fine-tuned to identify **525 bird species** from photographs. The model is based on **EfficientNet-B2** — a convolutional neural network architecture designed to be both accurate and computationally efficient. It was first pre-trained on **ImageNet** (1.2 million general images), then fine-tuned on ~84,000 bird photographs, achieving **99.1% test accuracy**.

This is the same **transfer learning** pattern we've seen throughout this course:
1. Pre-train on a large general dataset (ImageNet) → the model learns to recognize edges, textures, shapes
2. Fine-tune on a smaller domain-specific dataset (bird photos) → the model specializes in distinguishing species

**Why does this matter for conservation?**
- **Citizen science platforms** like eBird and iNaturalist rely on species identification at massive scale
- **Camera trap monitoring** generates millions of images that need automated classification
- **Biodiversity surveys** can be accelerated by AI-assisted species identification in the field

> 🏠 Model: [dennisjooo/Birds-Classifier-EfficientNetB2](https://huggingface.co/dennisjooo/Birds-Classifier-EfficientNetB2) (525 species, 99.1% accuracy)

## Load the model

In [ ]:
from transformers import pipeline
from PIL import Image
import requests
from io import BytesIO

# Load the bird classifier — same pipeline API as before, just a different task!
bird_classifier = pipeline(
    "image-classification",
    model="dennisjooo/Birds-Classifier-EfficientNetB2",
    device=0
)

# Check label mapping
num_species = len(bird_classifier.model.config.id2label)
print(f"✅ Bird species classifier loaded — can identify {num_species} species")

# Show a few example species the model knows
sample_species = [bird_classifier.model.config.id2label[i] for i in [0, 100, 200, 300, 400, 500]]
print(f"Sample species: {sample_species}")

## Helper function to load and display images

In [ ]:
import matplotlib.pyplot as plt

def load_image_from_url(url):
    """Download an image from a URL and return a PIL Image."""
    headers = {
        "User-Agent": "Mozilla/5.0 (Colab; Educational) Bird-Classifier-Demo/1.0"
    }
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()
    return Image.open(BytesIO(response.content)).convert("RGB")

def classify_and_show(image, title="Bird Photo", top_k=5):
    """Classify a bird image and display the results."""
    results = bird_classifier(image, top_k=top_k)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4),
                              gridspec_kw={'width_ratios': [1, 1.2]})

    # Show the image
    axes[0].imshow(image)
    axes[0].set_title(title, fontsize=12, fontweight='bold')
    axes[0].axis('off')

    # Show top predictions as horizontal bar chart
    labels = [r['label'].replace('_', ' ').title() for r in results]
    scores = [r['score'] for r in results]
    colors = ['#2d5016' if i == 0 else '#a8c686' for i in range(len(scores))]

    bars = axes[1].barh(range(len(labels)-1, -1, -1), scores, color=colors)
    axes[1].set_yticks(range(len(labels)-1, -1, -1))
    axes[1].set_yticklabels(labels)
    axes[1].set_xlabel("Confidence")
    axes[1].set_title("Top Predictions", fontsize=12, fontweight='bold')
    axes[1].set_xlim(0, 1)

    for i, (bar, score) in enumerate(zip(bars, scores)):
        axes[1].text(score + 0.01, bar.get_y() + bar.get_height()/2,
                     f'{score:.1%}', va='center', fontsize=10)

    plt.tight_layout()
    plt.show()

## Classify bird images from URLs

In [ ]:
# Bird images from public domain / Creative Commons sources
# Using species that exist in Israel and the Mediterranean region where possible

bird_urls = {
    "European Goldfinch": "https://upload.wikimedia.org/wikipedia/commons/7/7c/Carduelis_carduelis_close_up.jpg",
    "Common Kingfisher": "https://upload.wikimedia.org/wikipedia/commons/b/bc/Alcedo_atthis_-England-8_%28cropped%29.jpg",
    "Eurasian Hoopoe": "https://upload.wikimedia.org/wikipedia/commons/8/80/Eurasian_hoopoe_by_Gunjan_Pandey.jpg",
    "Bald Eagle": "https://upload.wikimedia.org/wikipedia/commons/1/1e/Bald_Eagle_Portrait.jpg",
    "Flamingo": "https://upload.wikimedia.org/wikipedia/commons/a/a9/Flamant_rose_Salines_de_Thyna.jpg",
}

import time

for bird_name, url in bird_urls.items():
    try:
        img = load_image_from_url(url)
        classify_and_show(img, title=f"Test: {bird_name}")
        time.sleep(4)  # Wait 2 seconds between requests to avoid rate limiting
    except Exception as e:
        print(f"⚠️ Could not load {bird_name}: {e}")
        print(f"   URL: {url}\n")